# Phase 6.5A — Unified TFL Agent (zero-call integration)

This notebook tests the unified Python interface in two modes: (1) exact frozen downstream replay of six development cases using the original twelve responses; (2) optional fresh PDF dry run through parsing, chunking, local hybrid retrieval and request preparation. Both modes prohibit new API attempts. Replay is not a fresh PDF end-to-end test. The live matching prompt is the simplified prospective protocol, whereas frozen replay uses the original D prompt and original fingerprints. Table targets only. All template and specification outputs remain candidates.

In [ ]:
from pathlib import Path
import sys, subprocess, importlib, json, shutil
REPO_URL='https://github.com/wohaofanaweishenme-cell/tfl-generation-agent.git'
BRANCH='simplified-rag-v3'
ROOT=Path('/content/tfl-simplified-rag-v3')
if not ROOT.exists():
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO_URL,str(ROOT)],check=True)
else:
    subprocess.run(['git','-C',str(ROOT),'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(ROOT),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{ROOT}[retrieval,test]'],check=True)
SRC=ROOT/'src'
sys.path.insert(0,str(SRC))
for name in list(sys.modules):
    if name=='tfl_agent' or name.startswith('tfl_agent.'):
        del sys.modules[name]
for p in (SRC/'tfl_agent').rglob('__pycache__'):
    shutil.rmtree(p)
importlib.invalidate_caches()
import tfl_agent
assert Path(tfl_agent.__file__).resolve().is_relative_to(SRC.resolve())
print('Commit:',subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip())
subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests/test_unified_pipeline.py')],check=True)

In [ ]:
from google.colab import drive
from datetime import datetime, timezone
from tfl_agent import TFLAgent
from tfl_agent.agent.unified_pipeline import ResponseExecutor,development_replay,summarize,write_json
from tfl_agent.agent.phase6_simplified_replay import ARCHIVE_NAME as MATCH_NAME,ARCHIVE_SHA as MATCH_SHA,load_archive
from tfl_agent.agent.phase6_specification_validation import ARCHIVE_NAME as SPEC_NAME,ARCHIVE_SHA as SPEC_SHA
from tfl_agent.agent.phase6_template_alignment import frozen
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive',timeout_ms=180000)
BASE=Path('/content/drive/MyDrive/Colab Notebooks/agent/midterm demo')
assert BASE.is_dir(), f'Missing project folder: {BASE}'
LIBRARY=ROOT/'generic_template_library'
def find_frozen(folder,name,expected):
    direct=BASE/folder/name
    candidates=[direct]+[p for p in (BASE/folder).glob('*.zip') if p!=direct]
    for p in candidates:
        if not p.is_file(): continue
        try:
            with frozen(p,name,expected): pass
            return p
        except (ValueError,OSError,KeyError): continue
    # Search same project only, for the exact inner filename in a different folder.
    for p in BASE.rglob(name):
        try:
            with frozen(p,name,expected): pass
            return p
        except (ValueError,OSError,KeyError): continue
    raise FileNotFoundError(f'Missing verified archive: {direct} (SHA256 {expected}). Restore the original ZIP; do not rerun LLM.')
MATCH=find_frozen('Phase633_Applicability_Pilot_Outputs',MATCH_NAME,MATCH_SHA)
SPEC=find_frozen('Phase64C_Specification_Pilot_Outputs',SPEC_NAME,SPEC_SHA)
RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT=Path('/content/Phase65A_Unified_Agent')/RUN_ID
OUT.mkdir(parents=True,exist_ok=False)
DEST=BASE/'Phase65A_Unified_Agent_Outputs'
DEST.mkdir(parents=True,exist_ok=True)
print('Verified matching archive:',MATCH)
print('Verified specification archive:',SPEC)
print('Library:',LIBRARY)
print('New LLM attempt budget: 0')

## Frozen downstream regression
The original matching and specification request fingerprints must match the verified raw records. No development annotations are loaded. Historical reference labels are used only after all predictions have been produced.

In [ ]:
import pandas as pd
from IPython.display import display
results,cache=development_replay(MATCH,SPEC)
summary=summarize(results)
assert len(results)==6 and len(cache)==12
assert summary['new_attempts']==0
assert not summary['generation_authorized']
assert all(r['mode']=='frozen_downstream_replay' for r in results)
# Evaluation-only join after inference.
_,_,labels,_=load_archive(MATCH)
comparison=[]
for r in results:
    rid=r['study_id']+'__'+r['entry_id']+'__D_applicability'
    expected=labels[rid]['expected'].split('|')
    baseline=(r['matching'] or {}).get('recommended_template')
    rec=r['recommended_template']
    comparison.append({'study_id':r['study_id'],'entry_id':r['entry_id'],'title':r['target']['title'],
                       'original':baseline,'aligned':rec,'expected_reference':'|'.join(expected),
                       'original_correct':baseline in expected,'aligned_correct':rec in expected,
                       'decision':r['stage_status']['alignment'],
                       'matching_protocol_valid':r['matching_protocol_valid'],
                       'candidate_fields':sum(f['status']=='candidate' for f in r['specification']['fields'].values())})
summary['development_reference_matches']={'original':sum(x['original_correct'] for x in comparison),
                                        'aligned':sum(x['aligned_correct'] for x in comparison),
                                        'n':len(comparison),'held_out':False}
summary['source_sha256']={'matching':MATCH_SHA,'specification':SPEC_SHA}
summary['fresh_pdf_test']='not_run'
write_json(OUT/'Frozen_Replay_Results.json',results)
pd.DataFrame(comparison).to_csv(OUT/'Replay_Comparison.csv',index=False,encoding='utf-8-sig')
write_json(OUT/'Summary.json',summary)
display(pd.DataFrame(comparison))
print(json.dumps(summary,indent=2))

## Optional real SAP PDF dry run
Set RUN_FRESH_PDF to True and supply an existing Drive PDF path relative to the project folder, target title, and verified physical PDF page range (1-based inclusive). Exclude appended reference TFL shells and content outside the SAP body. No upload is required. No claims will be fabricated on a cache miss; the notebook saves requests instead. This section may download public BGE model weights; embeddings are computed locally. The synthetic fresh-PDF test already ran above with a mock encoder/response; this section tests real retrieval on your PDF.

In [ ]:
RUN_FRESH_PDF=False #@param {type:"boolean"}
SAP_RELATIVE_PATH="" #@param {type:"string"}
TARGET_TITLE="" #@param {type:"string"}
BODY_FIRST=0 #@param {type:"integer"}
BODY_LAST=0 #@param {type:"integer"}

if RUN_FRESH_PDF:
    if not SAP_RELATIVE_PATH.strip() or not TARGET_TITLE.strip() or not 1<=BODY_FIRST<=BODY_LAST:
        raise ValueError('Set the existing Drive SAP relative path, title and verified physical page range.')
    sap=(BASE/SAP_RELATIVE_PATH).resolve()
    if not sap.is_relative_to(BASE.resolve()) or not sap.is_file() or sap.suffix.lower()!='.pdf':
        raise ValueError('SAP must be an existing PDF inside the project folder.')
    import torch
    from tfl_agent.tools.sap_hybrid import LocalEncoder
    encoder=LocalEncoder(device='cuda' if torch.cuda.is_available() else 'cpu')
    agent=TFLAgent(library_path=LIBRARY,model='deepseek-flash',encoder=encoder,
                   executor=ResponseExecutor(frozen=cache))
    fresh=agent.run(sap,TARGET_TITLE,body_pages=(BODY_FIRST,BODY_LAST))
    assert fresh['new_attempts']==0
    assert not fresh['generation_authorized']
    write_json(OUT/'Fresh_PDF_Result.json',fresh)
    summary['fresh_pdf_test']={'mode':'fresh_pdf','sap_sha256':fresh['sap_sha256'],
                               'target':fresh['target'],'retrieval':{k:fresh['retrieval'][k] for k in
                               ('body_pages','block_count','chunk_count','primary_ids','supplementary_ids')},
                               'stage_status':fresh['stage_status'],
                               'cache_sources':{k:v.get('cache_source') for k,v in fresh['responses'].items()},
                               'new_attempts':fresh['new_attempts'],
                               'complete_candidate_pipeline':fresh['complete_candidate_pipeline']}
    print(json.dumps(summary['fresh_pdf_test'],indent=2))
else:
    print('Fresh PDF dry run not selected. Replay alone does not validate fresh-PDF inference.')
write_json(OUT/'Summary.json',summary)

In [ ]:
saved=Path(shutil.make_archive(str(DEST/RUN_ID),'zip',root_dir=OUT))
print('Saved:',saved)
print('Files:',sorted(p.name for p in OUT.iterdir()))
print('New LLM attempts:',summary['new_attempts'])
print('Share only if authorized to disclose SAP excerpts and model responses.')

## Interpretation
Fresh preparation is independent of old experiment ZIPs; the frozen archives are needed only for the notebook replay and optional cache hits. Historical inputs were assembled with an earlier retrieval policy and matching prompt, so fresh fingerprints may differ. No fresh request is sent unless separately authorized in a bounded executor. The test suite includes synthetic API failure and fresh-PDF tests with mock responses, not actual remote model behavior. Body page selection remains a user-supplied boundary; no automatic shell exclusion beyond those bounds. Semantic applicability, completeness, figure/listing support, preset binding, DOCX output, and held-out evaluation remain outside this integration stage.